# **Aprendizaje Automático 1 - Trabajo Práctico 2**
## EDA - Predicción de lluvias en Australia

**Integrantes:** Alejandro Acosta, Diego Hauret, Gastón Rinaldi, Agustín Torres

Análisis exploratorio de datos previo a la construcción de un modelo de ...

**Indice:**
1. Carga y comprensión de los datos
2. Análisis de valores faltantes
3. Imputación de valores faltantes (ejemplo)
4. Análisis univariado (variable objetivo y predictoras)
5. Análisis bivariado (correlaciones y relación con MEDV)
6. Multicolinealidad entre predictoras
7. Detección de outliers
8. Conclusiones y próximos pasos


In [1]:
#from google.colab import drive
#drive.mount('/content/drive')

#%%capture
#!pip install shap

## Diccionario de variables

| Variable | Descripción |
| :--- | :--- |
| **Unnamed** | Índices |
| **Date** | Fecha |
| **Location** | Localidad de ubicación de la estación meteorológica |
| **MinTemp** | Temperatura mínima en °C |
| **MaxTemp** | Temperatura máxima en °C |
| **Rainfall** | Cantidad de lluvia registrada durante las últimas 24 horas (mm) |
| **Evaporation** | Evaporación aparente calculada (medida en mm) |
| **Sunshine** | Número de horas totales de sol brillante registradas a lo largo del día |
| **WindGustDir** | Dirección de la ráfaga de viento más fuerte |
| **WindGustSpeed** | Velocidad de la ráfaga de viento más fuerte |
| **WindDir9am** | Dirección del viento registrada a las 09:00 horas |
| **WindDir3pm** | Dirección del viento registrada a las 15:00 horas |
| **WindSpeed9am** | Velocidad del viento (en km/h). Promedio de los 10 minutos anteriores |
| **WindSpeed3pm** | Velocidad del viento (en km/h). Promedio de los 10 minutos anteriores |
| **Humidity9am** | Porcentaje de humedad relativa registrado a las 09:00 horas |
| **Humidity3pm** | Porcentaje de humedad relativa registrado a las 15:00 horas |
| **Pressure9am** | Presión atmosférica a las 09:00 horas (en hectopascales) |
| **Pressure3pm** | Presión atmosférica a las 15:00 horas (en hectopascales) |
| **Cloud9am** | Nubosidad a las 09:00 horas (0 indica cielo despejado y 8 nublado total) |
| **Cloud3pm** | Nubosidad a las 15:00 horas (0 indica cielo despejado y 8 nublado total) |
| **Temp9am** | Temperatura medida a las 09:00 horas en °C |
| **Temp3pm** | Temperatura medida a las 15:00 horas en °C |
| **RainToday** | Variable categórica binaria (Yes / No). Toma el valor Yes si la precipitación acumulada (Rainfall) en las últimas 24 horas hasta las 09:00 horas de día actual superó el 1.0 mm |
| **RainTomorrow** | Variable objetivo (target) a predecir (Yes / No). Indica si lloverá al día siguiente (las precipitaciones de mañana superarán el umbral de 1.0 mm) |

## 0. Carga de librerías

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
"""
import seaborn as sns
import shap
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, roc_auc_score, average_precision_score,
                             matthews_corrcoef, cohen_kappa_score, log_loss, brier_score_loss,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc,
                             precision_recall_curve, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV

#sns.set_theme(style="whitegrid")
"""

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Carga y comprensión de los datos

In [12]:
df = pd.read_csv('./weatherAUS_2026C2.csv')
#df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Aprendizaje_automatico/weatherAUS_2026C2.csv')
print("Dimensiones:", df.shape)
df.head(10)


Dimensiones: (145412, 25)


,Unnamed: 0,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,...,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow,RainfallTomorrow
0,0,2008-12-01,Albury,13.6000,22.5000,0.6000,NaN,NaN,W,45.0000,...,23.8000,1008.1000,1007.2000,8.0000,NaN,16.8000,20.8000,No,No,0.0000
1,1,2008-12-02,Albury,7.3000,25.8000,0.0000,NaN,NaN,WNW,44.0000,...,22.1000,1011.1000,1007.5000,NaN,NaN,18.3000,24.6000,No,No,0.0000
2,2,2008-12-03,Albury,13.2000,26.2000,0.0000,NaN,NaN,WSW,46.0000,...,31.5000,1007.3000,1009.4000,NaN,2.0000,21.2000,23.4000,No,No,0.0000
3,3,2008-12-04,Albury,10.0000,28.8000,0.0000,NaN,NaN,NE,23.0000,...,18.5000,1017.6000,1012.4000,NaN,NaN,17.4000,26.3000,No,No,1.0000
4,4,2008-12-05,Albury,17.4000,32.4000,1.0000,NaN,NaN,W,41.0000,...,34.6000,1010.9000,1005.9000,7.0000,8.0000,17.9000,29.1000,No,No,0.2000
5,5,2008-12-06,Albury,14.5000,30.1000,0.2000,NaN,NaN,WNW,55.0000,...,23.0000,1009.9000,1005.5000,NaN,NaN,20.9000,28.2000,No,No,0.0000
6,6,2008-12-07,Albury,15.1000,25.3000,0.0000,NaN,NaN,W,51.0000,...,20.3000,1009.3000,1008.5000,1.0000,NaN,18.3000,24.0000,No,No,0.0000
7,7,2008-12-08,Albury,8.1000,26.1000,0.0000,NaN,NaN,W,34.0000,...,20.8000,1013.5000,1009.7000,NaN,NaN,17.1000,25.9000,No,No,0.0000
8,8,2008-12-09,Albury,9.5000,31.8000,0.0000,NaN,NaN,NNW,80.0000,...,8.5000,1008.9000,1003.6000,NaN,NaN,18.0000,28.9000,No,Yes,1.4000
9,9,2008-12-10,Albury,13.4000,30.1000,1.4000,NaN,NaN,W,28.0000,...,24.6000,1007.2000,1005.7000,NaN,NaN,18.9000,28.2000,Yes,No,0.0000


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145412 entries, 0 to 145411
Data columns (total 25 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        145412 non-null  int64  
 1   Date              145412 non-null  object 
 2   Location          145412 non-null  object 
 3   MinTemp           143928 non-null  float64
 4   MaxTemp           144159 non-null  float64
 5   Rainfall          142152 non-null  float64
 6   Evaporation       82658 non-null   float64
 7   Sunshine          75616 non-null   float64
 8   WindGustDir       135096 non-null  object 
 9   WindGustSpeed     135159 non-null  float64
 10  WindDir9am        134850 non-null  object 
 11  WindDir3pm        141186 non-null  object 
 12  WindSpeed9am      143645 non-null  float64
 13  WindSpeed3pm      142351 non-null  float64
 14  Humidity9am       142759 non-null  float64
 15  Humidity3pm       140907 non-null  float64
 16  Pressure9am       13

## 2. Observaciones sobre las columnas del archivo


* `Unnamed: 0`: es un índice que no aporta información, entonces se elimina.
* `RainfallTomorrow`: es la **cantidad de lluvia (mm) de mañana**. Es una fuga de información (*data leakage*): RainTomorrow se define justamente como "mañana llovió ≥ 1 mm", por lo que esta columna casi determina el target. En un caso real **no la conoceríamos al momento de predecir**. Se elimina de los predictores.

In [13]:
df = df.drop(columns=["Unnamed: 0"])
df = df.sort_values(["Date"]).reset_index(drop=True)
df["Date"] = pd.to_datetime(df["Date"])

num_cols = df.select_dtypes(include="number").columns.drop("RainfallTomorrow").tolist()
cat_cols = ["Location", "WindGustDir", "WindDir9am", "WindDir3pm", "RainToday"]

df = df.drop(columns=["RainfallTomorrow"])
print("Período:", df["Date"].min().date(), "->", df["Date"].max().date())
print("Ubicaciones de las estaciones:", df["Location"].nunique())
print("Filas duplicadas:", df.duplicated().sum())
print("\nVariables numéricas:", num_cols)
print("Variables categóricas:", cat_cols)

Período: 2007-11-01 -> 2017-06-24
Ubicaciones de las estaciones: 49
Filas duplicadas: 0

Variables numéricas: ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm']
Variables categóricas: ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday']


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145412 entries, 0 to 145411
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   Date           145412 non-null  datetime64[ns]
 1   Location       145412 non-null  object        
 2   MinTemp        143928 non-null  float64       
 3   MaxTemp        144159 non-null  float64       
 4   Rainfall       142152 non-null  float64       
 5   Evaporation    82658 non-null   float64       
 6   Sunshine       75616 non-null   float64       
 7   WindGustDir    135096 non-null  object        
 8   WindGustSpeed  135159 non-null  float64       
 9   WindDir9am     134850 non-null  object        
 10  WindDir3pm     141186 non-null  object        
 11  WindSpeed9am   143645 non-null  float64       
 12  WindSpeed3pm   142351 non-null  float64       
 13  Humidity9am    142759 non-null  float64       
 14  Humidity3pm    140907 non-null  float64       
 15  

## 3. Análisis de valores faltantes en columna objetivo


Revisamos cuántos nulos tiene cada columna y si hay algún patrón (filas con muchos nulos a la vez, o nulos concentrados).


In [17]:
# Se eliminan las filas sin target o sin RainToday (se hace ANTES de dividir: no usa información de otras filas)
n0 = len(df)
df = df.dropna(subset=["RainTomorrow", "RainToday"]).copy()
print(f"Filas eliminadas: {n0 - len(df)} ({(n0 - len(df)) / n0:.2%}) | Filas restantes: {len(df)}")

Filas eliminadas: 4665 (3.21%) | Filas restantes: 140747


## 4. Implementación de un modelo de AutoML con Pycaret

In [ ]:
# 1. Importar el módulo de clasificación de PyCaret
from pycaret.classification import *

# A. IMPORTANTE: Ordenamos el dataframe por la columna de fecha ya que se trata de
# una serie de tiempo y PyCaret necesita que los datos estén ordenados cronológicamente
# para poder hacer la división Train/Test correctamente.
df = df.sort_values(by='Date').reset_index(drop=True)

# B. Configurar PyCaret indicando la estructura de Serie de Tiempo
# PyCaret imputará nulos, codificará variables categóricas y dividirá los datos (Train/Test)
exp_lluvia = setup(
    data=df,
    target='RainTomorrow',
    session_id=RANDOM_STATE,
    data_split_shuffle=False,
    data_split_stratify=False,  # Desactiva la estratificación
    fold_strategy='timeseries',
    fold=5,
    fix_imbalance=True,
    normalize=True
)

# 3. Entrenar y comparar todos los modelos
# PyCaret probará algoritmos como XGBoost, LightGBM, Random Forest, etc.
# Te mostrará una tabla ordenada por la mejor métrica (Accuracy, AUC, Recall...)
print("Entrenando modelos...")
mejor_modelo = compare_models()

# 4. Ajustar el modelo (Tuning de hiperparámetros)
# Optimiza el algoritmo ganador para extraer su máximo rendimiento
print("Optimizando modelos...")
modelo_optimizado = tune_model(mejor_modelo)

# 5. Evaluar el modelo (Gráficas e interpretabilidad)
# Abre una interfaz para ver la Matriz de Confusión, Curva ROC e Importancia de Variables
print("Evaluando mejor modelo...")
evaluate_model(modelo_optimizado)

# 6. Finalizar el modelo
# Entrena el algoritmo optimizado utilizando el 100% de los datos de tu dataframe df
print("Evaluando modelo final...")
modelo_final = finalize_model(modelo_optimizado)

# 7. Guardar el pipeline completo
# Genera un archivo 'pipeline_lluvia.pkl' con todo el preprocesamiento y el modelo listo
print("Guardando modelo...")
save_model(modelo_final, 'pipeline_lluvia')


,Description,Value
0,Session id,42
1,Target,RainTomorrow
2,Target type,Binary
3,Target mapping,"No: 0, Yes: 1"
4,Original data shape,"(140747, 23)"
5,Transformed data shape,"(195343, 70)"
6,Transformed train set shape,"(153118, 70)"
7,Transformed test set shape,"(42225, 70)"
8,Numeric features,16
9,Date features,1


Entrenando modelos...


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.8498,0.8748,0.8498,0.8415,0.8426,0.5283,0.5353,24.6420
et,Extra Trees Classifier,0.8480,0.8728,0.8480,0.8385,0.8370,0.5052,0.5192,21.1520
gbc,Gradient Boosting Classifier,0.8424,0.8669,0.8424,0.8366,0.8386,0.5244,0.5263,51.7300
ridge,Ridge Classifier,0.8322,0.8675,0.8322,0.8340,0.8328,0.5193,0.5199,3.1080
lda,Linear Discriminant Analysis,0.8322,0.8675,0.8322,0.8340,0.8328,0.5193,0.5199,3.6880
lr,Logistic Regression,0.8321,0.8682,0.8321,0.8345,0.8330,0.5207,0.5215,7.3040
ada,Ada Boost Classifier,0.8262,0.8445,0.8262,0.8190,0.8210,0.4700,0.4735,13.4500
svm,SVM - Linear Kernel,0.8187,0.8592,0.8187,0.8287,0.8221,0.4985,0.5019,3.7700
dummy,Dummy Classifier,0.7768,0.5000,0.7768,0.6036,0.6793,0.0000,0.0000,2.7480
dt,Decision Tree Classifier,0.7695,0.6869,0.7695,0.7789,0.7737,0.3596,0.3605,6.2100


Optimizando modelos...


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8138,0.8531,0.8138,0.8231,0.8176,0.4982,0.4999
1,0.8081,0.8599,0.8081,0.8190,0.8123,0.5132,0.5155
2,0.8286,0.8634,0.8286,0.8342,0.8311,0.5069,0.5076
3,0.8355,0.8694,0.8355,0.8417,0.8382,0.5222,0.5231
4,0.8231,0.8515,0.8231,0.8286,0.8256,0.4819,0.4824
Mean,0.8218,0.8595,0.8218,0.8293,0.8250,0.5045,0.5057
Std,0.0099,0.0066,0.0099,0.0080,0.0092,0.0138,0.0140


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).
Evaluando mejor modelo...


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

Evaluando modelo final...
Guardando modelo...
Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('label_encoding',
                  TransformerWrapperWithInverse(exclude=None, include=None,
                                                transformer=LabelEncoder())),
                 ('date_feature_extractor',
                  TransformerWrapper(exclude=None, include=['Date'],
                                     transformer=ExtractDateTimeFeatures(features=['day',
                                                                                   'month',
                                                                                   'year']))),
                 ('numerical_imputer',
                  TransformerWrapper(exclude=None,
                                     inc...
                  RandomForestClassifier(bootstrap=True, ccp_alpha=0.0,
                                         class_weight=None, criterion='gini',
                                         max_depth=None, max_features='sqrt',
                  